In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgecommercedev1", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

ecommerce stgecommercedev1 ecomm-raw-data


In [0]:
# readChangeFeed flag is used to read the change feed (_change_type column mainly)
df = spark.readStream \
.format("delta") \
.option("readChangeFeed", "true") \
.table(f"{catalog_name}.silver.slv_order_returns")

#df.limit(10).display()

In [0]:
df_union = df.filter("_change_type IN ('insert', 'update_postimage')")

In [0]:
df_union = df_union.withColumn("date_id", F.date_format(F.col("order_dt"), "yyyyMMdd").cast(IntegerType()))

df_union = df_union.withColumn("return_days", F.datediff(F.col("return_ts"), F.col("order_dt")))

df_union = df_union.withColumn("within_policy", F.when(F.col("return_days") <= 15, F.lit(1)).otherwise(F.lit(0)))

df_union = df_union.withColumn("is_late_return", F.when(F.col("return_days") > 15, F.lit(1)).otherwise(F.lit(0)))

In [0]:
returns_gold_df = df_union.select(
    F.col("date_id"),
    F.col("return_ts"),
    F.col("order_dt"),
    F.col("order_id"),
    F.col("reason"),
    F.col("return_days"),
    F.col("within_policy"),
    F.col("is_late_return")
)

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_returns/"
print(gold_checkpoint_path)

def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_order_returns"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.order_id = batch_table.order_id",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

returns_gold_df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()

abfss://ecomm-raw-data@stgecommercedev1.dfs.core.windows.net/checkpoint/gold/fact_order_returns/


In [0]:
spark.sql(f"SELECT max(order_dt) FROM {catalog_name}.gold.gld_fact_order_returns").show()

+-------------+
|max(order_dt)|
+-------------+
|   2025-09-24|
+-------------+

